# Tratamento dos Dados — Mortalidade 

---
## 1. Imports

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# Caminhos
PASTA = Path("/home/anna/tcc/Limpeza e Integração/Mortalidade/Cidades Atendidas por Cada Regional")
SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Mortalidade")
ARQUIVO = SAIDA / "tabela_mortes_por_faixa_etária.csv"

ARQUIVOS_MUNICIPIOS = [
    Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv"),
]

# Período de análise
ANOS = list(range(1979, 2025))

# Faixas etárias padronizadas
FAIXAS_ETARIAS = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70 OU MAIS",
]

# Filtro de sexo: 'mulheres', 'homens' ou None (todos os arquivos)
SEXO_DESEJADO: str | None = "mulheres"

# CID-10 de interesse
CID_PREFERIDO = "c50"  # neoplasia maligna da mama

# Padrões de texto compilados
PADRAO_ANO_ARQUIVO = re.compile(r"^(19\d{2}|20\d{2})")
PADRAO_INTERVALO = re.compile(r"(\d{1,3})\s*(?:a|-|ate|até)\s*(\d{1,3})", re.IGNORECASE)
PADRAO_OU_MAIS = re.compile(r"^(\d{1,3})\s*(?:ou\s*mais|\+|e\s*mais|99\+)$", re.IGNORECASE)

SAIDA.mkdir(parents=True, exist_ok=True)
print(f"Pasta de saída: {SAIDA}")
print(f"   Período: {ANOS[0]}–{ANOS[-1]} | Sexo: {SEXO_DESEJADO} | CID: {CID_PREFERIDO.upper()}")

Pasta de saída: /home/anna/tcc/Seleção e Transformação/Mortalidade
   Período: 1979–2024 | Sexo: mulheres | CID: C50


---
## 3. Funções de Normalização de Texto

Funções para padronizar rótulos de faixa etária, nomes de municípios e parsing de intervalos de idade.

In [3]:
def normalizar_texto(texto: str) -> str:
    """Remove acentos, converte para minúsculas e normaliza espaços e hífens."""
    texto_limpo = unicodedata.normalize("NFKD", str(texto).strip())
    texto_limpo = "".join(caractere for caractere in texto_limpo if not unicodedata.combining(caractere))
    texto_limpo = texto_limpo.lower().replace("_", " ").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", texto_limpo).strip()


def remover_sufixo_mg(texto: str) -> str:
    """Remove ' - MG' e variações do final do nome do município."""
    texto_limpo = normalizar_texto(texto)
    texto_limpo = re.sub(r"\s*-\s*mg$", "", texto_limpo)
    texto_limpo = re.sub(r"\s+mg$", "", texto_limpo)
    return texto_limpo.strip()


def normalizar_chave_municipio(texto: str) -> str:
    """Normaliza o município para chave de junção: sem sufixo MG, hífens e apóstrofos, e com aliases resolvidos."""
    texto_limpo = remover_sufixo_mg(texto)
    texto_limpo = texto_limpo.replace("-", " ").replace("'", " ")
    texto_limpo = re.sub(r"\s+", " ", texto_limpo).strip()
    aliases = {
        "dona eusebia": "dona euzebia",
        "sao thome das letras": "sao tome das letras",
        "brasopolis": "brazopolis",
    }
    return aliases.get(texto_limpo, texto_limpo)


---
## 4. Funções de Parsing de Faixa Etária

Converte rótulos de coluna dos CSVs (ex: "60 a 64", "70 ou mais") para as 8 faixas padronizadas.

In [4]:
def _faixa_por_idade_inicial(idade_inicial: int) -> str | None:
    """Retorna a faixa padronizada a partir da idade inicial de um intervalo."""
    if idade_inicial < 0:
        return None
    if idade_inicial < 10:
        return "0-9"
    if idade_inicial < 20:
        return "10-19"
    if idade_inicial < 30:
        return "20-29"
    if idade_inicial < 40:
        return "30-39"
    if idade_inicial < 50:
        return "40-49"
    if idade_inicial < 60:
        return "50-59"
    if idade_inicial < 70:
        return "60-69"
    return "70 OU MAIS"


def _extrair_intervalo_idade(rotulo: str) -> tuple[int, int] | None:
    """Extrai (idade_inicial, idade_final) do rótulo de coluna; None para rótulos não numéricos."""
    rotulo_normalizado = normalizar_texto(rotulo)
    IGNORAR = {
        "cid", "ci", "total", "ignorado", "ign", "obito", "óbito",
        "obitos", "óbitos", "subtotal", "sem informacao", "sem informação",
        "ign.", "ign",
    }
    if not rotulo_normalizado or rotulo_normalizado in IGNORAR or not rotulo_normalizado[0].isdigit():
        return None

    correspondencia_ou_mais = PADRAO_OU_MAIS.search(rotulo_normalizado)
    if correspondencia_ou_mais:
        return (int(correspondencia_ou_mais.group(1)), 999)

    correspondencia_intervalo = PADRAO_INTERVALO.search(rotulo_normalizado)
    if correspondencia_intervalo:
        idade_inicial = int(correspondencia_intervalo.group(1))
        idade_final = int(correspondencia_intervalo.group(2))
        if idade_final >= 99:
            idade_final = 999
        return (idade_inicial, idade_final)

    if rotulo_normalizado.isdigit():
        idade = int(rotulo_normalizado)
        return (idade, idade)

    return None


def mapear_faixa_etaria(rotulo: str) -> str | None:
    """Mapeia o rótulo bruto para a faixa etária padrão, rejeitando intervalos maiores que 10 anos."""
    intervalo = _extrair_intervalo_idade(rotulo)
    if intervalo is None:
        return None
    idade_inicial, idade_final = intervalo
    if idade_final < idade_inicial:
        return None
    if idade_final != 999 and (idade_final - idade_inicial) > 10:
        return None
    faixa = _faixa_por_idade_inicial(idade_inicial)
    if faixa in set(FAIXAS_ETARIAS):
        return faixa
    return None


---
## 5. Funções de Leitura dos CSVs Brutos

Os CSVs do DATASUS/SIM têm estrutura não padronizada: linhas de metadados no topo,
cabeçalho iniciando com "CID;" e valores numéricos separados por ";".
As funções abaixo localizam automaticamente o cabeçalho e a linha de dados do CID de interesse.

In [5]:
def ler_linhas_csv(caminho_csv: Path) -> list[str]:
    """Lê as linhas de um CSV tentando latin-1 primeiro, depois utf-8."""
    for codificacao in ("latin-1", "utf-8"):
        try:
            with open(caminho_csv, "r", encoding=codificacao, errors="replace") as arquivo_aberto:
                return [linha_bruta.rstrip("\n") for linha_bruta in arquivo_aberto]
        except Exception:
            continue
    return []


def detectar_sexo_arquivo(caminho_csv: Path) -> str | None:
    """Detecta o sexo referenciado no arquivo lendo as 3 primeiras linhas."""
    linhas = ler_linhas_csv(caminho_csv)
    topo = " ".join(normalizar_texto(linha_bruta) for linha_bruta in linhas[:3] if linha_bruta.strip())
    if "em homens" in topo:
        return "homens"
    if "em mulheres" in topo:
        return "mulheres"
    return None


def dividir_linha_csv(linha: str) -> list[str]:
    """Divide linha CSV por ';' e remove células vazias do final."""
    partes = [celula.strip() for celula in str(linha).split(";")]
    while partes and partes[-1] == "":
        partes.pop()
    return partes


def extrair_pares_cabecalho_valor(caminho_csv: Path, cid_preferido: str = CID_PREFERIDO) -> list[tuple[str, str]]:
    """Devolve os pares (rótulo da coluna, valor) da linha de dados do CID de interesse."""
    linhas = ler_linhas_csv(caminho_csv)
    indice_cabecalho = None
    for indice, linha_bruta in enumerate(linhas):
        if re.match(r"^\s*CID\s*;", linha_bruta, re.IGNORECASE):
            indice_cabecalho = indice
            break
    if indice_cabecalho is None or indice_cabecalho + 1 >= len(linhas):
        return []

    cabecalho = dividir_linha_csv(linhas[indice_cabecalho])
    candidatos = []
    for linha_bruta in linhas[indice_cabecalho + 1:]:
        if ";" not in linha_bruta:
            continue
        celulas = dividir_linha_csv(linha_bruta)
        if len(celulas) >= 2 and any(celula.strip() for celula in celulas[1:]):
            candidatos.append(linha_bruta)
    if not candidatos:
        return []

    # Prefere a linha que menciona o CID de interesse; senão, a primeira
    linha_dados = candidatos[0]
    for linha_bruta in candidatos:
        primeira_celula = normalizar_texto(dividir_linha_csv(linha_bruta)[0])
        if re.search(rf"\b{re.escape(cid_preferido.lower())}\b", primeira_celula):
            linha_dados = linha_bruta
            break

    valores = dividir_linha_csv(linha_dados)
    pares = []
    for indice in range(len(cabecalho)):
        valor = ""
        if indice < len(valores):
            valor = valores[indice]
        pares.append((cabecalho[indice], valor))
    return pares


def para_inteiro(valor) -> int | None:
    """Converte texto numérico (com ponto ou vírgula) para int; None se inválido ou ausente (zero é preservado)."""
    if valor is None:
        return None
    texto_limpo = str(valor).strip().replace(".", "").replace(",", "")
    if not texto_limpo or not texto_limpo.isdigit():
        return None
    return int(texto_limpo)


---
## 6. Consolidação da Mortalidade

`consolidar_mortalidade` percorre a árvore de diretórios `regional → município → CSVs`,
lê cada arquivo anual, extrai os pares (faixa etária, valor) e acumula por município.

In [6]:
def consolidar_mortalidade(pasta_raiz: Path) -> pd.DataFrame:
    """Consolida os CSVs por regional e município em uma linha por município, com colunas AAAA_faixa."""
    registros = []

    pastas_regionais = []
    for pasta in pasta_raiz.iterdir():
        if pasta.is_dir():
            pastas_regionais.append(pasta)

    for regional_dir in sorted(pastas_regionais):
        regional = regional_dir.name

        pastas_municipios = []
        for pasta in regional_dir.iterdir():
            if pasta.is_dir():
                pastas_municipios.append(pasta)

        for municipio_dir in sorted(pastas_municipios):
            municipio = municipio_dir.name
            registro: dict[str, object] = {
                "Município": municipio,
                "Regional de Saúde": regional,
            }

            for caminho_csv in sorted(municipio_dir.glob("*.csv")):
                if SEXO_DESEJADO is not None:
                    sexo = detectar_sexo_arquivo(caminho_csv)
                    if sexo is not None and sexo != SEXO_DESEJADO:
                        continue

                ano = None
                correspondencia = PADRAO_ANO_ARQUIVO.search(caminho_csv.name)
                if correspondencia:
                    ano_candidato = int(correspondencia.group(1))
                    if ano_candidato in ANOS:
                        ano = ano_candidato
                if ano is None:
                    continue

                pares = extrair_pares_cabecalho_valor(caminho_csv)
                for rotulo, valor_raw in pares:
                    faixa = mapear_faixa_etaria(rotulo)
                    if faixa is None:
                        continue
                    valor = para_inteiro(valor_raw)
                    if valor is None:
                        continue
                    coluna = f"{ano}_{faixa}"
                    anterior = registro.get(coluna)
                    if anterior is None:
                        registro[coluna] = valor
                    else:
                        registro[coluna] = int(anterior) + valor

            registros.append(registro)

    return pd.DataFrame(registros)


---
## 7. Organização de Colunas e Enriquecimento com Código IBGE

`organizar_colunas` garante que todas as combinações `YYYY_faixa` existam (com `NaN` onde não há dado),
calcula `Total_YYYY` por ano (renomeado para `YYYY_TOTAL` na etapa de salvar) e reordena as colunas
na ordem canônica.

`carregar_mapa_codigos` e `adicionar_codigo_municipio` fazem o join com a tabela de municípios
para adicionar o código IBGE de 7 dígitos.

In [7]:
def organizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Garante as colunas AAAA_faixa, calcula Total_AAAA por ano e reordena as colunas."""
    colunas_ano_faixa = []
    for ano in ANOS:
        for faixa in FAIXAS_ETARIAS:
            colunas_ano_faixa.append(f"{ano}_{faixa}")
    colunas_total_ano = [f"Total_{ano}" for ano in ANOS]

    faltantes = []
    for coluna in colunas_ano_faixa:
        if coluna not in df.columns:
            faltantes.append(coluna)
    if faltantes:
        df = pd.concat(
            [df, pd.DataFrame(np.nan, index=df.index, columns=faltantes)],
            axis=1,
        )

    for coluna in colunas_ano_faixa:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").astype("Int64")

    for ano in ANOS:
        colunas = [f"{ano}_{faixa}" for faixa in FAIXAS_ETARIAS]
        df[f"Total_{ano}"] = df[colunas].sum(axis=1, min_count=1).astype("Int64")

    colunas_fixas = ["Município", "Regional de Saúde"]
    colunas_conhecidas = {*colunas_fixas, *colunas_ano_faixa, *colunas_total_ano}
    extras = []
    for coluna in df.columns:
        if coluna not in colunas_conhecidas:
            extras.append(coluna)
    colunas_finais = colunas_fixas + colunas_ano_faixa + extras + colunas_total_ano

    return (
        df[colunas_finais]
        .sort_values(["Regional de Saúde", "Município"], na_position="last")
        .reset_index(drop=True)
    )


def carregar_mapa_codigos() -> dict[str, str]:
    """Carrega o mapa MUNICÍPIO NORMALIZADO -> CÓDIGO IBGE do primeiro arquivo existente de ARQUIVOS_MUNICIPIOS."""
    arquivo = None
    for candidato in ARQUIVOS_MUNICIPIOS:
        if candidato.exists():
            arquivo = candidato
            break
    if arquivo is None:
        candidatos = ", ".join(str(candidato) for candidato in ARQUIVOS_MUNICIPIOS)
        raise FileNotFoundError(f"Arquivo de municípios não encontrado. Tentativas: {candidatos}")

    referencia = pd.read_csv(arquivo, dtype={"CÓDIGO IBGE": "string"})
    obrigatorias = {"MUNICÍPIO NORMALIZADO", "CÓDIGO IBGE"}
    faltantes = obrigatorias - set(referencia.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes: {sorted(faltantes)}")

    referencia = referencia.dropna(subset=["MUNICÍPIO NORMALIZADO", "CÓDIGO IBGE"]).copy()
    referencia["MUNICÍPIO NORMALIZADO"] = referencia["MUNICÍPIO NORMALIZADO"].astype(str).map(normalizar_chave_municipio)
    referencia["CÓDIGO IBGE"] = referencia["CÓDIGO IBGE"].astype(str).str.strip()
    referencia = referencia.drop_duplicates(subset=["MUNICÍPIO NORMALIZADO"], keep="first")

    return dict(zip(referencia["MUNICÍPIO NORMALIZADO"], referencia["CÓDIGO IBGE"]))


def adicionar_codigo_municipio(df: pd.DataFrame, mapa_codigos: dict[str, str]) -> pd.DataFrame:
    """Insere a coluna 'Código' após 'Município' e lista os municípios sem correspondência."""
    if "Município" not in df.columns:
        raise ValueError("DataFrame não possui a coluna 'Município'.")

    chaves = df["Município"].astype(str).map(normalizar_chave_municipio)
    codigos = chaves.map(mapa_codigos).astype("string")

    sem_codigo = df.loc[codigos.isna(), "Município"].astype(str).dropna().sort_values().unique().tolist()
    if sem_codigo:
        print(f"⚠️  {len(sem_codigo)} município(s) sem correspondência de código:")
        for municipio_nome in sem_codigo:
            print(f"   - {municipio_nome}")
    else:
        print("✅ Todos os municípios tiveram código IBGE encontrado.")

    df = df.drop(columns=["CÓDIGO", "Código"], errors="ignore").copy()
    posicao = list(df.columns).index("Município") + 1
    df.insert(posicao, "Código", codigos)
    return df

---
## 8. Pipeline de Tratamento


In [8]:
print("Verificando estrutura de diretórios...")
regionais = []
for pasta in PASTA.iterdir():
    if pasta.is_dir():
        regionais.append(pasta)

municipios_total = 0
for regional in regionais:
    for pasta in regional.iterdir():
        if pasta.is_dir():
            municipios_total += 1
print(f"   Regionais encontradas : {len(regionais)}")
print(f"   Municípios encontrados: {municipios_total}")

Verificando estrutura de diretórios...
   Regionais encontradas : 29
   Municípios encontrados: 853


In [9]:
# Etapa 1: Consolidação
print("\nEtapa 1: Consolidando mortalidade...")
df = consolidar_mortalidade(PASTA)
print(f"   Shape após consolidação: {df.shape}")


Etapa 1: Consolidando mortalidade...


   Shape após consolidação: (853, 290)


In [10]:
# Etapa 2: Organização de colunas e totais
print("\nEtapa 2: Organizando colunas e calculando totais...")
df = organizar_colunas(df)
print(f"   Shape após organização: {df.shape}")


Etapa 2: Organizando colunas e calculando totais...
   Shape após organização: (853, 416)


In [11]:
# Etapa 3: Normalização e código IBGE
print("\nEtapa 3: Adicionando código IBGE...")
df["Município"] = df["Município"].map(remover_sufixo_mg)
mapa_codigos = carregar_mapa_codigos()
df = adicionar_codigo_municipio(df, mapa_codigos)


Etapa 3: Adicionando código IBGE...


✅ Todos os municípios tiveram código IBGE encontrado.


In [12]:
# Etapa 4: Salvar
print("\n💾 Etapa 4: Salvando CSV...")

# Nomes finais das colunas: MUNICÍPIO, CÓDIGO IBGE, REGIONAL DE SAÚDE, AAAA_TOTAL
renomeacao_final = {
    "Município": "MUNICÍPIO",
    "Código": "CÓDIGO IBGE",
    "Regional de Saúde": "REGIONAL DE SAÚDE",
}
for ano in ANOS:
    renomeacao_final[f"Total_{ano}"] = f"{ano}_TOTAL"
df = df.rename(columns=renomeacao_final)
df.to_csv(ARQUIVO, sep=",", index=False, encoding="utf-8")

colunas_total = []
for coluna in df.columns:
    if coluna.endswith("_TOTAL"):
        colunas_total.append(coluna)
soma_total = int(df[colunas_total].sum(skipna=True).sum())

print("\n╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo  : {str(ARQUIVO.name):<41}║")
print(f"║  Linhas   : {len(df):<41}║")
print(f"║  Colunas  : {len(df.columns):<41}║")
print(f"║  Municípios únicos  : {df['MUNICÍPIO'].nunique():<30}║")
print(f"║  Regionais únicas   : {df['REGIONAL DE SAÚDE'].nunique(dropna=True):<30}║")
print(f"║  Soma de todas YYYY_TOTAL : {soma_total:<24}║")
print("╚══════════════════════════════════════════════════════╝")


💾 Etapa 4: Salvando CSV...

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo  : tabela_mortes_por_faixa_etária.csv       ║
║  Linhas   : 853                                      ║
║  Colunas  : 417                                      ║
║  Municípios únicos  : 853                           ║
║  Regionais únicas   : 28                            ║
║  Soma de todas YYYY_TOTAL : 42422                   ║
╚══════════════════════════════════════════════════════╝


---
## 9. Inspeção Pós-Processamento

Verificação da tabela final: shape, amostra, valores ausentes e estatísticas do total.

In [13]:
display(Markdown(f"**Shape final:** {df.shape[0]} linhas × {df.shape[1]} colunas"))

colunas_total = []
for coluna in df.columns:
    if coluna.endswith("_TOTAL"):
        colunas_total.append(coluna)
colunas_previa = ["MUNICÍPIO", "CÓDIGO IBGE", "REGIONAL DE SAÚDE"] + colunas_total[-5:]
display(Markdown("**Amostra (últimas colunas de total):**"))
display(df[colunas_previa].head(5))

# Estatísticas do total por município
total_por_municipio = df[colunas_total].sum(axis=1, skipna=True)
total_geral = int(total_por_municipio.sum())
media = total_por_municipio.mean()
mediana = total_por_municipio.median()
desvio = total_por_municipio.std()
print(f"\nTotal de mortes no período : {total_geral:,}".replace(",", "."))
print(f"Média por município        : {media:,.1f}".replace(",", "."))
print(f"Mediana por município      : {mediana:,.1f}".replace(",", "."))
print(f"Desvio padrão              : {desvio:,.1f}".replace(",", "."))

sem_codigo_ibge = df["CÓDIGO IBGE"].isna().sum()
if sem_codigo_ibge > 0:
    display(Markdown(f"⚠️ **{sem_codigo_ibge} município(s) sem código IBGE:**"))
    display(df.loc[df["CÓDIGO IBGE"].isna(), ["MUNICÍPIO", "REGIONAL DE SAÚDE"]].head(10))
else:
    display(Markdown("✅ **Todos os municípios possuem código IBGE.**"))

**Shape final:** 853 linhas × 417 colunas

**Amostra (últimas colunas de total):**

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL
0,alfenas,3101607,Alfenas,7,9,6,11,7
1,alterosa,3102001,Alfenas,2,2,1,4,1
2,arceburgo,3104106,Alfenas,1,1,<NA>,2,<NA>
3,areado,3104304,Alfenas,1,1,<NA>,1,1
4,bandeira do sul,3105301,Alfenas,<NA>,2,1,<NA>,<NA>



Total de mortes no período : 42.422
Média por município        : 49.7
Mediana por município      : 11.0
Desvio padrão              : 331.9


✅ **Todos os municípios possuem código IBGE.**

---
## 10. Análise da Regional de Divinópolis

Filtra os municípios pertencentes à regional de Divinópolis e soma os óbitos por ano e por faixa etária.

In [14]:
REGIONAL_ALVO = "Divinópolis"

df_divinopolis = df[df["REGIONAL DE SAÚDE"] == REGIONAL_ALVO].copy()

municipios_divinopolis = sorted(df_divinopolis["MUNICÍPIO"].unique())
print(f"Municípios da regional de {REGIONAL_ALVO} ({len(municipios_divinopolis)}):")
for municipio_nome in municipios_divinopolis:
    print(f"   - {municipio_nome}")

Municípios da regional de Divinópolis (55):
   - aguanil
   - araujos
   - arcos
   - bambui
   - bom despacho
   - bom sucesso
   - camacho
   - campo belo
   - cana verde
   - candeias
   - carmo da mata
   - carmo do cajuru
   - carmopolis de minas
   - claudio
   - conceicao do para
   - corrego danta
   - corrego fundo
   - cristais
   - divinopolis
   - dores do indaia
   - estrela do indaia
   - formiga
   - igaratinga
   - iguatama
   - itaguara
   - itapecerica
   - itatiaiucu
   - itauna
   - japaraiba
   - lagoa da prata
   - leandro ferreira
   - luz
   - martinho campos
   - medeiros
   - moema
   - nova serrana
   - oliveira
   - onca de pitangui
   - pains
   - para de minas
   - passa tempo
   - pedra do indaia
   - perdigao
   - pimenta
   - piracema
   - pitangui
   - santana do jacare
   - santo antonio do amparo
   - santo antonio do monte
   - sao francisco de paula
   - sao goncalo do para
   - sao jose da varginha
   - sao sebastiao do oeste
   - serra da saudade

In [15]:
# Soma por ano (todas as faixas e municípios da regional)
colunas_total_ano = [f"{ano}_TOTAL" for ano in ANOS]
soma_por_ano = df_divinopolis[colunas_total_ano].sum(skipna=True)
soma_por_ano.index = ANOS
soma_por_ano.index.name = "ANO"
soma_por_ano.name = "ÓBITOS"

display(Markdown(f"**Óbitos por ano — Regional de {REGIONAL_ALVO}:**"))
display(soma_por_ano.to_frame())

# Soma por faixa etária (todos os anos e municípios da regional)
somas_por_faixa = {}
for faixa in FAIXAS_ETARIAS:
    colunas_da_faixa = [f"{ano}_{faixa}" for ano in ANOS]
    somas_por_faixa[faixa] = df_divinopolis[colunas_da_faixa].sum(skipna=True).sum()
soma_por_faixa = pd.Series(somas_por_faixa, name="ÓBITOS")

soma_por_faixa.index.name = "FAIXA ETÁRIA"

display(Markdown(f"**Óbitos por faixa etária — Regional de {REGIONAL_ALVO}:**"))
display(soma_por_faixa.to_frame())

print(f"\nTotal geral ({REGIONAL_ALVO}): {int(soma_por_ano.sum()):,}".replace(",", "."))

**Óbitos por ano — Regional de Divinópolis:**

,ÓBITOS
ANO,
1979,24
1980,19
1981,25
1982,18
1983,25
1984,23
1985,26
1986,25
1987,39


**Óbitos por faixa etária — Regional de Divinópolis:**

,ÓBITOS
FAIXA ETÁRIA,
0-9,0
10-19,1
20-29,17
30-39,166
40-49,468
50-59,623
60-69,589
70 OU MAIS,820



Total geral (Divinópolis): 2.684
